In [1]:
! nvidia-smi

Tue Oct 13 16:13:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.91.07              Driver Version: 595.91.07      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   61C    P3             15W /   55W |      15MiB /   8188MiB |     12%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import sys
from pathlib import Path
import io, zipfile, urllib.request

need_data = not Path("data/ecommerce/gt_images.json").exists()
need_req = not Path("requirements.txt").exists()
if need_data or need_req:
    url = "https://codeload.github.com/ranjanrakesh47g/ObjectDetection/zip/refs/heads/main"
    with zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read())) as z:
        root = next(n for n in z.namelist() if n.endswith("data/ecommerce/gt_images.json")).split("data/")[0]
        names = []
        if need_req:
            names.append(root + "requirements.txt")
        if need_data:
            names += [n for n in z.namelist() if n.startswith(root + "data/") and not n.endswith("/")]
        for name in names:
            dest = Path(name[len(root):])
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(z.read(name))

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("pip", "install -q -r requirements.txt")
    if not Path("/tmp/colab_deps_restarted").exists():
        Path("/tmp/colab_deps_restarted").write_text("1")
        print("Restarting the runtime so NumPy reloads.")
        get_ipython().kernel.do_shutdown(restart=True)

In [3]:
import base64, json, time
from enum import Enum
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict, Field
from tqdm import tqdm
import pandas as pd

In [4]:
load_dotenv()

True

In [5]:
def labels(name, values):
    return Enum(name, {value: value for value in values})

def detection_model(vocab):
    class ObjectDetectionOutput(BaseModel):
        model_config = ConfigDict(extra="forbid")
        relation: labels("relation", vocab["relation"])
        foreground: labels("foreground", vocab["foreground"])
        background: labels("background", vocab["background"])
    return ObjectDetectionOutput

def load_preds():
    path = Path("results/predictions.csv")
    path.parent.mkdir(exist_ok=True)
    cols = ["image_filepath", "gt_foreground", "gt_background", "gt_relation"]
    if not path.exists():
        pd.DataFrame(columns=cols).to_csv(path, index=False)
    return pd.read_csv(path)

def save_preds(model, frame):
    frame = frame.rename(columns={"image": "image_filepath"}).set_index("image_filepath")
    frame.columns = [f"{model}_{c}" for c in frame.columns]
    gt = []
    for split in ("ecommerce", "news"):
        for row in json.loads(Path(f"data/{split}/gt_images.json").read_text()):
            gt.append({"image_filepath": f"data/{split}/images/{row['file']}", "gt_foreground": row["foreground"], "gt_background": row["background"], "gt_relation": row["relation"]})
    gt = pd.DataFrame(gt).set_index("image_filepath")
    old = load_preds().set_index("image_filepath")
    old = old.drop(columns=[c for c in old.columns if c in frame.columns or c in gt.columns])
    pd.concat([gt, old, frame], axis=1).to_csv("results/predictions.csv")
    save_metrics(model)

def save_metrics(model):
    from sklearn.metrics import accuracy_score, f1_score
    df = pd.read_csv("results/predictions.csv").dropna(subset=[f"{model}_latency"])
    row = {"model_name": model}
    total = 0
    all_gold, all_pred = [], []
    for split, prefix in (("ecommerce", "ecom"), ("news", "news")):
        part = df[df.image_filepath.str.startswith(f"data/{split}/")]
        correct, golds, preds = 0, [], []
        for short, key in (("fg", "foreground"), ("bg", "background"), ("rel", "relation")):
            gold, pred = part[f"gt_{key}"], part[f"{model}_{key}"].fillna("")
            correct += gold.eq(pred).sum()
            labels = sorted(set(gold))
            row[f"{prefix}_{short}_acc"] = round(accuracy_score(gold, pred), 3)
            row[f"{prefix}_{short}_micro_f1"] = round(f1_score(gold, pred, average="micro", labels=labels, zero_division=0), 3)
            row[f"{prefix}_{short}_macro_f1"] = round(f1_score(gold, pred, average="macro", labels=labels, zero_division=0), 3)
            golds += [f"{short}:{v}" for v in gold]
            preds += [f"{short}:{v}" for v in pred]
        labels = sorted(set(golds))
        row[f"{prefix}_acc"] = round(correct / (len(part) * 3), 3)
        row[f"{prefix}_micro_f1"] = round(f1_score(golds, preds, average="micro", labels=labels, zero_division=0), 3)
        row[f"{prefix}_macro_f1"] = round(f1_score(golds, preds, average="macro", labels=labels, zero_division=0), 3)
        total += correct
        all_gold += golds
        all_pred += preds
    labels = sorted(set(all_gold))
    row["acc"] = round(total / (len(df) * 3), 3)
    row["micro_f1"] = round(f1_score(all_gold, all_pred, average="micro", labels=labels, zero_division=0), 3)
    row["macro_f1"] = round(f1_score(all_gold, all_pred, average="macro", labels=labels, zero_division=0), 3)
    row["latency (s)"] = round(df[f"{model}_latency"].mean(), 3)
    path = Path("results/metrics.csv")
    cols = ["model_name"] + [f"{p}_{t}_{s}" for p in ("ecom", "news") for t in ("fg", "bg", "rel") for s in ("acc", "micro_f1", "macro_f1")] + ["ecom_acc", "ecom_micro_f1", "ecom_macro_f1", "news_acc", "news_micro_f1", "news_macro_f1", "acc", "micro_f1", "macro_f1", "latency (s)"]
    old = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=cols)
    hit = old.model_name == model
    if hit.any():
        old.loc[hit, list(row)] = list(row.values())
    else:
        old = pd.concat([old, pd.DataFrame([row])], ignore_index=True)
    old = old.reindex(columns=cols)
    old.to_csv(path, index=False)
    per_image = {"qwen3.8-27b_openrouter": 0.00016, "qwen3-vl-30b-a3b_openrouter": 0.000279, "cloud-vision_google": 0.00875, "gemini-3.1-flash-lite_google": 0.000485}
    out = old[["model_name", "acc", "micro_f1", "macro_f1", "latency (s)"]].copy()
    out.insert(1, "source", ["open-source" if "openrouter" not in n and "google" not in n else "commercial" for n in out.model_name])
    out["cost ($/1k images)"] = [round(per_image.get(name, 0) * 1000, 3) for name in out.model_name]
    out.to_csv(path.parent / "overall_metrics.csv", index=False)

## OpenRouter: Qwen3.8-27B, Qwen3.8-30b-a3b-instruct

In [6]:
import os
import httpx

def predict_openrouter(paths, vocab):
    Output = detection_model(vocab)

    class Batch(BaseModel):
        model_config = ConfigDict(extra="forbid")
        labels: list[Output] = Field(min_length=len(paths), max_length=len(paths))

    text = (
        "Label this image. When two labels both fit, choose the more specific one.\n"
        f"foreground: {', '.join(vocab['foreground'])}\n"
        f"background: {', '.join(vocab['background'])}\n"
        f"relation: {', '.join(vocab['relation'])}"
    )
    content = [{"type": "text", "text": text}]
    for path in paths:
        image = base64.b64encode(path.read_bytes()).decode()
        content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image}"}})
    body = {
        # "model": "qwen/qwen3.8-27b",
        "model": "qwen/qwen3-vl-30b-a3b-instruct",
        "messages": [{"role": "user", "content": content}],
        "response_format": {
            "type": "json_schema",
            "json_schema": {"name": "labels", "strict": True, "schema": Batch.model_json_schema()},
        },
        "reasoning": {"effort": "none"},
        "temperature": 0,
        "max_completion_tokens": 256,
    }
    reply = httpx.post(
        "https://openrouter.ai/api/v1/chat/completions",
        json=body,
        headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"},
        timeout=120,
    )
    reply.raise_for_status()
    reply = reply.json()
    return Batch.model_validate_json(reply["choices"][0]["message"]["content"]).model_dump(mode="json")["labels"]


In [7]:
batch_size = 1
# model = "qwen3.8-27b_openrouter"
model = "qwen3-vl-30b-a3b_openrouter"
load_preds()
results, logged = [], []
for split in ("ecommerce", "news"):
    rows = json.loads(Path(f"data/{split}/gt_images.json").read_text())
    vocab = {k: sorted({r[k] for r in rows}) for k in ("foreground", "background", "relation")}
    groups = [rows[i:i + batch_size] for i in range(0, len(rows), batch_size)]
    for group in tqdm(groups, desc=split):
        paths = [Path(f"data/{split}/images") / row["file"] for row in group]
        t0 = time.perf_counter()
        preds = predict_openrouter(paths, vocab)
        sec = (time.perf_counter() - t0) / len(group)
        for path, row, pred in zip(paths, group, preds):
            results.append((split, row, pred))
            logged.append({"image": path.as_posix(), "foreground": pred["foreground"], "background": pred["background"], "relation": pred["relation"], "latency": round(sec, 3)})
            tqdm.write(f"{split} {row['file']} {pred} {sec:.2f}s")
preds_df = pd.DataFrame(logged)
print(f"{preds_df['latency'].mean():.2f}s/image")
save_preds(model, preds_df)


ecommerce:   0%|          | 0/38 [00:00<?, ?it/s]

ecommerce:   3%|▎         | 1/38 [00:06<03:45,  6.09s/it]

ecommerce A_black_coffee_mug_on_an_office_desk_with_a_blurred_laptop_in_the_background.jpg {'relation': 'object_with_background', 'foreground': 'coffee_mug', 'background': 'blurred_laptop'} 6.09s


ecommerce:   5%|▌         | 2/38 [00:08<02:16,  3.78s/it]

ecommerce a_camera_is_attached_to_a_tripod_with_a_mountain_in_the_background.jpg {'relation': 'object_with_background', 'foreground': 'camera', 'background': 'mountain'} 2.17s


...

news Joe_Biden_at_State_of_the_Union.jpg {'relation': 'people_at_location', 'foreground': 'Joe_Biden', 'background': 'Oval_Office'} 3.97s


news:  93%|█████████▎| 14/15 [00:52<00:03,  3.28s/it]

news Joe_Biden_at_White_House_Correspondents_Dinner.jpg {'relation': 'people_at_event', 'foreground': 'Joe_Biden', 'background': 'White_House_Correspondents_Dinner'} 4.24s


news: 100%|██████████| 15/15 [01:13<00:00,  4.93s/it]


news Mr_Bezos_at_Florida.jpg {'relation': 'people_at_event', 'foreground': 'Jeff_Bezos', 'background': 'Argo_press_junket'} 20.98s
6.43s/image


## Clarifai: Gemini 2.5 Flash


In [ ]:
import os, urllib.request

def predict_clarifai(path, vocab):
    Output = detection_model(vocab)
    image = base64.b64encode(path.read_bytes()).decode()
    text = (
        "Label this image. When two labels both fit, choose the more specific one.\n"
        f"foreground: {', '.join(vocab['foreground'])}\n"
        f"background: {', '.join(vocab['background'])}\n"
        f"relation: {', '.join(vocab['relation'])}"
    )
    body = {
        "model": "https://clarifai.com/gcp/generate/models/gemini-2_5-flash",
        "messages": [{"role": "user", "content": [
            {"type": "text", "text": text},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image}"}},
        ]}],
        "response_format": {
            "type": "json_schema",
            "json_schema": {"name": "labels", "strict": True, "schema": Output.model_json_schema()},
        },
        "temperature": 0,
        "max_completion_tokens": 128,
    }
    req = urllib.request.Request(
        "https://api.clarifai.com/v2/ext/openai/v1/chat/completions",
        data=json.dumps(body).encode(),
        headers={"Authorization": f"Bearer {os.environ['CLARIFAI_PAT']}", "Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req) as res:
        reply = json.loads(res.read())
    return Output.model_validate_json(reply["choices"][0]["message"]["content"]).model_dump(mode="json")

In [ ]:
model = "gemini-2.5-flash_clarifai"
load_preds()
results, logged = [], []
for split in ("ecommerce", "news"):
    rows = json.loads(Path(f"data/{split}/gt_images.json").read_text())
    vocab = {k: sorted({r[k] for r in rows}) for k in ("foreground", "background", "relation")}
    for row in tqdm(rows, desc=split):
        path = Path(f"data/{split}/images") / row["file"]
        t0 = time.perf_counter()
        pred = predict_clarifai(path, vocab)
        sec = time.perf_counter() - t0
        results.append((split, row, pred))
        logged.append({"image": path.as_posix(), "foreground": pred["foreground"], "background": pred["background"], "relation": pred["relation"], "latency": round(sec, 3)})
        tqdm.write(f"{split} {row['file']} {pred} {sec:.2f}s")
preds_df = pd.DataFrame(logged)
print(f"{preds_df['latency'].mean():.2f}s/image")
save_preds(model, preds_df)

## Google Cloud Vision


In [6]:
import os, re, urllib.request
from dotenv import load_dotenv
load_dotenv(".env")

ALIASES = {
    "coffee_mug": ["coffee mug", "coffee cup", "mug"],
    "cup": ["cup"],
    "sunglasses": ["sunglasses", "glasses", "eyewear"],
    "camera": ["camera", "dslr", "slr"],
    "phone": ["smartphone", "cellphone", "mobile phone", "phone"],
    "laptop": ["laptop"],
    "blurred_laptop": ["laptop", "computer", "monitor", "screen"],
    "blurred_lights": ["bokeh", "lights", "light", "night"],
    "mountain": ["mountain", "snow", "peak"],
    "ocean": ["ocean", "sea", "beach"],
    "man": ["man", "boy", "male"],
    "woman": ["woman", "girl", "lady", "female"],
    "people": ["people", "crowd", "person"],
    "urban_architecture": ["architecture", "skyline", "cityscape", "building"],
    "bookshelf": ["bookshelf", "bookcase"],
    "flowers": ["flower"],
    "dessert": ["dessert", "cake", "pastry"],
    "water_bottle": ["water bottle"],
    "perfume_bottle": ["perfume", "fragrance"],
    "flower_vase": ["vase"],
    "alarm_clock": ["alarm clock", "clock"],
    "handbag": ["handbag", "purse"],
    "kettle": ["kettle", "teapot"],
    "pen": ["fountain pen", "pen"],
    "lamp": ["lamp"],
    "book": ["book"],
    "soap": ["soap"],
    "jar": ["jar"],
    "towel": ["towel"],
    "horse": ["horse"],
    "Barack_Obama": ["barack obama", "obama"],
    "Donald_Trump": ["donald trump", "trump"],
    "Joe_Biden": ["joe biden", "biden"],
    "Ben_Affleck": ["ben affleck", "affleck"],
    "Brad_Pitt": ["brad pitt"],
    "Jeff_Bezos": ["jeff bezos", "bezos"],
    "White_House": ["white house"],
    "Oval_Office": ["oval office"],
    "United_Nations": ["united nations"],
    "Venice_Film_Festival": ["venice film"],
    "White_House_Correspondents_Dinner": ["correspondents dinner"],
    "State_of_the_Union": ["state of the union"],
    "Panama_City": ["panama"],
    "Antalya": ["antalya"],
    "Florida": ["florida"],
    "Buffalo": ["buffalo"],
}

def has(text, key):
    return re.search(rf"(?<![a-z]){re.escape(key)}s?(?![a-z])", text) is not None

def pick(names, choices):
    for name in names:
        low = name.lower()
        found, n = "", 0
        for choice in choices:
            keys = ALIASES.get(choice, [choice.lower().replace("_", " ")])
            for key in keys:
                if has(low, key) and len(key) > n:
                    found, n = choice, len(key)
        if found:
            return found
    return ""

def predict_vision(path, vocab):
    image = base64.b64encode(path.read_bytes()).decode()
    body = {"requests": [{"image": {"content": image}, "features": [
        {"type": "OBJECT_LOCALIZATION", "maxResults": 8},
        {"type": "LABEL_DETECTION", "maxResults": 15},
        {"type": "WEB_DETECTION", "maxResults": 15},
        {"type": "LANDMARK_DETECTION", "maxResults": 5},
    ]}]}
    req = urllib.request.Request(
        "https://vision.googleapis.com/v1/images:annotate?key=" + os.environ["GOOGLE_API_KEY"],
        data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req) as res:
        ann = json.loads(res.read())["responses"][0]
    def area(obj):
        vs = obj["boundingPoly"]["normalizedVertices"]
        xs = [v.get("x", 0) for v in vs]
        ys = [v.get("y", 0) for v in vs]
        return (max(xs) - min(xs)) * (max(ys) - min(ys))

    objects = [o["name"] for o in sorted(ann.get("localizedObjectAnnotations", []), key=area)]
    labels = [o["description"] for o in ann.get("labelAnnotations", [])]
    web = ann.get("webDetection", {})
    labels += [e["description"] for e in web.get("webEntities", []) if e.get("description")]
    labels += [g["label"] for g in web.get("bestGuessLabels", [])]
    labels += [p.get("pageTitle", "") for p in web.get("pagesWithMatchingImages", [])]
    labels += [o["description"] for o in ann.get("landmarkAnnotations", [])]
    foreground = pick(objects, vocab["foreground"]) or pick(labels, vocab["foreground"])
    background = pick([n for n in labels if n], [c for c in vocab["background"] if c != foreground])
    relation = pick(objects + labels, vocab["relation"])
    if not relation and len(vocab["relation"]) == 1:
        relation = vocab["relation"][0]
    elif not relation and pick(objects + labels, ["people"]):
        blob = " ".join(objects + labels).lower()
        relation = "people_at_event" if re.search(r"event|festival|dinner|ceremony|party|speech|convention", blob) else "people_at_location"
    return {"foreground": foreground, "background": background, "relation": relation}



In [7]:
model = "cloud-vision_google"
load_preds()
results, logged = [], []
for split in ("ecommerce", "news"):
    rows = json.loads(Path(f"data/{split}/gt_images.json").read_text())
    vocab = {k: sorted({r[k] for r in rows}) for k in ("foreground", "background", "relation")}
    for row in tqdm(rows, desc=split):
        path = Path(f"data/{split}/images") / row["file"]
        t0 = time.perf_counter()
        pred = predict_vision(path, vocab)
        sec = time.perf_counter() - t0
        results.append((split, row, pred))
        logged.append({"image": path.as_posix(), "foreground": pred["foreground"], "background": pred["background"], "relation": pred["relation"], "latency": round(sec, 3)})
        tqdm.write(f"{split} {row['file']} {pred} {sec:.2f}s")
preds_df = pd.DataFrame(logged)
print(f"{preds_df['latency'].mean():.2f}s/image")
save_preds(model, preds_df)


ecommerce:   3%|▎         | 1/38 [00:01<00:42,  1.14s/it]

ecommerce A_black_coffee_mug_on_an_office_desk_with_a_blurred_laptop_in_the_background.jpg {'foreground': 'coffee_mug', 'background': 'blurred_laptop', 'relation': 'object_with_background'} 1.14s


ecommerce:   5%|▌         | 2/38 [00:01<00:33,  1.07it/s]

ecommerce a_camera_is_attached_to_a_tripod_with_a_mountain_in_the_background.jpg {'foreground': 'camera', 'background': 'mountain', 'relation': 'object_with_background'} 0.78s


ecommerce:   8%|▊         | 3/38 [00:02<00:34,  1.02it/s]

...

news Joe_Biden_at_State_of_the_Union.jpg {'foreground': '', 'background': '', 'relation': 'people_at_event'} 0.88s


news:  93%|█████████▎| 14/15 [00:11<00:00,  1.21it/s]

news Joe_Biden_at_White_House_Correspondents_Dinner.jpg {'foreground': '', 'background': '', 'relation': 'people_at_event'} 0.57s


news: 100%|██████████| 15/15 [00:13<00:00,  1.15it/s]


news Mr_Bezos_at_Florida.jpg {'foreground': 'Jeff_Bezos', 'background': '', 'relation': 'people_at_event'} 1.28s
1.02s/image


## Gemini: 3.1 Flash-Lite


In [6]:
import os, time, urllib.request, urllib.error

def predict_gemini(path, vocab):
    Output = detection_model(vocab)
    image = base64.b64encode(path.read_bytes()).decode()
    text = (
        "Label this image. When two labels both fit, choose the more specific one.\n"
        f"foreground: {', '.join(vocab['foreground'])}\n"
        f"background: {', '.join(vocab['background'])}\n"
        f"relation: {', '.join(vocab['relation'])}"
    )
    schema = {
        "type": "object",
        "properties": {key: {"type": "string", "enum": vocab[key]} for key in vocab},
        "required": list(vocab),
    }
    body = {
        "contents": [{"role": "user", "parts": [
            {"text": text},
            {"inline_data": {"mime_type": "image/jpeg", "data": image}},
        ]}],
        "generationConfig": {
            "temperature": 0,
            "maxOutputTokens": 256,
            "thinkingConfig": {"thinkingBudget": 0},
            "responseMimeType": "application/json",
            "responseSchema": schema,
        },
    }
    req = urllib.request.Request(
        "https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent?key=" + os.environ["GEMINI_API_KEY"],
        data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
    )
    for attempt in range(8):
        try:
            with urllib.request.urlopen(req, timeout=45) as res:
                reply = json.loads(res.read())
            break
        except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError) as err:
            busy = isinstance(err, (urllib.error.URLError, TimeoutError)) or err.code in (429, 503)
            if not busy or attempt == 7:
                raise
            time.sleep(35 if getattr(err, "code", None) == 429 else 2 * (attempt + 1))
    content = reply["candidates"][0]["content"]["parts"][0]["text"]
    return Output.model_validate_json(content).model_dump(mode="json")

model = "gemini-3.1-flash-lite_google"
load_preds()
results, logged = [], []
for split in ("ecommerce", "news"):
    rows = json.loads(Path(f"data/{split}/gt_images.json").read_text())
    vocab = {k: sorted({r[k] for r in rows}) for k in ("foreground", "background", "relation")}
    for row in tqdm(rows, desc=split):
        path = Path(f"data/{split}/images") / row["file"]
        t0 = time.perf_counter()
        pred = predict_gemini(path, vocab)
        sec = time.perf_counter() - t0
        results.append((split, row, pred))
        logged.append({"image": path.as_posix(), "foreground": pred["foreground"], "background": pred["background"], "relation": pred["relation"], "latency": round(sec, 3)})
        tqdm.write(f"{split} {row['file']} {pred} {sec:.2f}s")
        time.sleep(6)
preds_df = pd.DataFrame(logged)
print(f"{preds_df['latency'].mean():.2f}s/image")
save_preds(model, preds_df)

ecommerce:   0%|          | 0/38 [00:06<?, ?it/s]

ecommerce A_black_coffee_mug_on_an_office_desk_with_a_blurred_laptop_in_the_background.jpg {'relation': 'object_with_background', 'foreground': 'coffee_mug', 'background': 'blurred_laptop'} 6.31s


ecommerce:   3%|▎         | 1/38 [00:18<07:35, 12.31s/it]

ecommerce a_camera_is_attached_to_a_tripod_with_a_mountain_in_the_background.jpg {'relation': 'object_with_background', 'foreground': 'camera', 'background': 'mountain'} 5.84s


ecommerce:   5%|▌         | 2/38 [00:30<07:13, 12.04s/it]

...

news Joe_Biden_at_White_House_Correspondents_Dinner.jpg {'relation': 'people_at_event', 'foreground': 'Joe_Biden', 'background': 'White_House_Correspondents_Dinner'} 6.39s


news:  93%|█████████▎| 14/15 [02:58<00:12, 12.77s/it]

news Mr_Bezos_at_Florida.jpg {'relation': 'people_at_location', 'foreground': 'Jeff_Bezos', 'background': 'Florida'} 6.91s


news: 100%|██████████| 15/15 [03:04<00:00, 12.31s/it]


7.48s/image


## Local GPU: Qwen3-VL-2B, 4-bit, batch of 2

In [6]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

MODEL = "Qwen/Qwen3-VL-2B-Instruct"
local_model = AutoModelForImageTextToText.from_pretrained(
    MODEL,
    cache_dir="models",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL, cache_dir="models", max_pixels=401408)
processor.tokenizer.padding_side = "left"

/home/rakesh/Downloads/ShortTok/3.ObjectDetection/ObjectDetection/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 625/625 [00:04<00:00, 126.61it/s]


In [7]:
def pick(value, choices):
    text = str(value).strip()
    if text in choices:
        return text
    for part in text.replace(",", " ").split():
        if part in choices:
            return part
    return ""

def predict_local(paths, vocab):
    prompt = (
        'Reply with only {"foreground":"...","background":"...","relation":"..."}. One label each, no commas inside values. When two labels both fit, choose the more specific one.\n'
        f"foreground: {', '.join(vocab['foreground'])}\n"
        f"background: {', '.join(vocab['background'])}\n"
        f"relation: {', '.join(vocab['relation'])}"
    )
    messages = [[{
        "role": "user",
        "content": [
            {"type": "image", "image": str(path.resolve())},
            {"type": "text", "text": prompt},
        ],
    }] for path in paths]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt", padding=True,
    ).to(local_model.device)
    with torch.inference_mode():
        generated = local_model.generate(
            **inputs, max_new_tokens=160, do_sample=False,
            stop_strings=["}"], tokenizer=processor.tokenizer,
        )
    texts = processor.batch_decode(
        [out[len(src):] for src, out in zip(inputs.input_ids, generated)],
        skip_special_tokens=True,
    )
    preds = []
    for text in texts:
        if text.count('"') % 2:
            text += '"'
        if not text.rstrip().endswith("}"):
            text += "}"
        start, end = text.find("{"), text.rfind("}")
        try:
            obj = json.loads(text[start:end + 1])
        except json.JSONDecodeError:
            obj = {}
        preds.append({key: pick(obj.get(key, text), vocab[key]) for key in vocab})
    return preds

In [8]:
batch_size = 2
model = "qwen3-vl-2b_4bit_local"
load_preds()
results, logged = [], []
for split in ("ecommerce", "news"):
    rows = json.loads(Path(f"data/{split}/gt_images.json").read_text())
    vocab = {k: sorted({r[k] for r in rows}) for k in ("foreground", "background", "relation")}
    groups = [rows[i:i + 2] for i in range(0, len(rows), batch_size)]
    for group in tqdm(groups, desc=split):
        paths = [Path(f"data/{split}/images") / row["file"] for row in group]
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        preds = predict_local(paths, vocab)
        torch.cuda.synchronize()
        sec = (time.perf_counter() - t0) / len(group)
        for path, row, pred in zip(paths, group, preds):
            results.append((split, row, pred))
            logged.append({"image": path.as_posix(), "foreground": pred["foreground"], "background": pred["background"], "relation": pred["relation"], "latency": round(sec, 3)})
            tqdm.write(f"{split} {row['file']} {pred} {sec:.2f}s")
preds_df = pd.DataFrame(logged)
print(f"{preds_df['latency'].mean():.2f}s/image")
save_preds(model, preds_df)

ecommerce:   0%|          | 0/19 [00:00<?, ?it/s][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
/home/rakesh/Downloads/ShortTok/3.ObjectDetection/ObjectDetection/.venv/lib/python3.12/site-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
ecommerce:   5%|▌         | 1/19 [00:02<00:42,  2.35s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


ecommerce A_black_coffee_mug_on_an_office_desk_with_a_blurred_laptop_in_the_background.jpg {'foreground': 'coffee_mug', 'background': '', 'relation': 'object_with_background'} 1.17s
ecommerce a_camera_is_attached_to_a_tripod_with_a_mountain_in_the_background.jpg {'foreground': 'camera', 'background': 'urban_architecture', 'relation': 'object_with_background'} 1.17s


ecommerce:  11%|█         | 2/19 [00:03<00:29,  1.73s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


ecommerce a_camera_on_a_tripod_with_a_mountain_in_the_background.jpg {'foreground': 'camera', 'background': 'mountain', 'relation': 'object_with_background'} 0.65s
ecommerce A_close_up_shot_of_a_fountain_pen_with_an_ink_bottle_highlighted_by_warm_bokeh_lights_in_the_background.jpg {'foreground': 'pen', 'background': 'blurred_lights', 'relation': 'object_with_background'} 0.65s


ecommerce:  16%|█▌        | 3/19 [00:04<00:24,  1.55s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


...

news Joe_Biden_at_Buffalo.jpg {'foreground': 'Joe_Biden', 'background': 'White_House', 'relation': 'people_at_event'} 0.67s
news Joe_Biden_at_Oval_Office.jpg {'foreground': 'Joe_Biden', 'background': 'White_House', 'relation': 'people_at_location'} 0.67s


news:  88%|████████▊ | 7/8 [00:12<00:01,  1.66s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


news Joe_Biden_at_State_of_the_Union.jpg {'foreground': 'Joe_Biden', 'background': 'White_House', 'relation': 'people_at_event'} 0.62s
news Joe_Biden_at_White_House_Correspondents_Dinner.jpg {'foreground': 'Joe_Biden', 'background': 'White_House', 'relation': 'people_at_location'} 0.62s


news: 100%|██████████| 8/8 [00:13<00:00,  1.68s/it]

news Mr_Bezos_at_Florida.jpg {'foreground': 'Jeff_Bezos', 'background': 'White_House', 'relation': 'people_at_event'} 1.05s
0.74s/image


In [9]:
from sklearn.metrics import classification_report

def pick(value, choices):
    text = str(value).strip()
    if text in choices:
        return text
    for part in text.replace(",", " ").split():
        if part in choices:
            return part
    return ""

keys = ("foreground", "background", "relation")
for split in ("ecommerce", "news"):
    pairs = [(pred, row) for name, row, pred in results if name == split]
    labels = {key: sorted({row[key] for _, row in pairs}) for key in keys}
    pairs = [({key: pick(pred.get(key, ""), labels[key]) for key in keys}, row) for pred, row in pairs]
    exact = sum(all(pred[k] == row[k] for k in keys) for pred, row in pairs) / len(pairs)
    accuracy = sum(pred[k] == row[k] for pred, row in pairs for k in keys) / (len(pairs) * len(keys))
    print(split, "exact", round(exact, 3), "accuracy", round(accuracy, 3))
    for key in keys:
        gold = [row[key] for _, row in pairs]
        pred = [item[key] for item, _ in pairs]
        print(key)
        print(classification_report(gold, pred, labels=labels[key], zero_division=0))

ecommerce exact 0.658 accuracy 0.886
foreground
                precision    recall  f1-score   support

   alarm_clock       1.00      1.00      1.00         1
      backpack       1.00      1.00      1.00         1
          book       1.00      1.00      1.00         1
        camera       1.00      1.00      1.00         9
    coffee_mug       0.89      1.00      0.94         8
           cup       0.00      0.00      0.00         1
   flower_vase       1.00      1.00      1.00         1
       handbag       1.00      1.00      1.00         1
           jar       1.00      1.00      1.00         1
        kettle       1.00      1.00      1.00         1
          lamp       1.00      1.00      1.00         1
        laptop       1.00      1.00      1.00         1
           pen       1.00      1.00      1.00         1
perfume_bottle       1.00      1.00      1.00         1
         phone       1.00      1.00      1.00         2
          soap       1.00      1.00      1.00         1

## Colab GPU: Qwen3-VL-4B, structured output

In [9]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

COLAB_MODEL = "Qwen/Qwen3-VL-4B-Instruct"
colab_model = AutoModelForImageTextToText.from_pretrained(
    COLAB_MODEL, cache_dir="models", dtype=torch.float16, device_map="auto",
)
colab_processor = AutoProcessor.from_pretrained(COLAB_MODEL, cache_dir="models", max_pixels=401408)
colab_processor.tokenizer.padding_side = "left"

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/64.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [10]:
def predict_colab(paths, vocab):
    Output = detection_model(vocab)
    tools = [{
        "type": "function",
        "function": {
            "name": "label_image",
            "description": "Choose one foreground, one background, and one relation.",
            "parameters": Output.model_json_schema(),
        },
    }]
    messages = [[{
        "role": "user",
        "content": [
            {"type": "image", "image": str(path.resolve())},
            {"type": "text", "text": (
                "Label this image. When two labels both fit, choose the more specific one.\n"
                f"foreground: {', '.join(vocab['foreground'])}\n"
                f"background: {', '.join(vocab['background'])}\n"
                f"relation: {', '.join(vocab['relation'])}"
            )},
        ],
    }] for path in paths]
    inputs = colab_processor.apply_chat_template(
        messages, tools=tools, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt", padding=True,
    ).to(colab_model.device)
    with torch.inference_mode():
        generated = colab_model.generate(
            **inputs, max_new_tokens=180, do_sample=False,
            stop_strings=["</tool_call>"], tokenizer=colab_processor.tokenizer,
        )
    texts = colab_processor.batch_decode(
        [out[len(src):] for src, out in zip(inputs.input_ids, generated)],
        skip_special_tokens=True,
    )
    preds = []
    for text in texts:
        blob = text[text.find("{"):text.rfind("}") + 1]
        try:
            obj = json.loads(blob).get("arguments", {})
            if isinstance(obj, str):
                obj = json.loads(obj)
            preds.append(Output.model_validate(obj).model_dump(mode="json"))
        except Exception:
            preds.append({key: "" for key in ("foreground", "background", "relation")})
    return preds

In [11]:
batch_size = 2
model = "qwen3-vl-4b_colab"
load_preds()
results, logged = [], []
for split in ("ecommerce", "news"):
    rows = json.loads(Path(f"data/{split}/gt_images.json").read_text())
    vocab = {k: sorted({r[k] for r in rows}) for k in ("foreground", "background", "relation")}
    groups = [rows[i:i + 2] for i in range(0, len(rows), batch_size)]
    for group in tqdm(groups, desc=split):
        paths = [Path(f"data/{split}/images") / row["file"] for row in group]
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        preds = predict_colab(paths, vocab)
        torch.cuda.synchronize()
        sec = (time.perf_counter() - t0) / len(group)
        for path, row, pred in zip(paths, group, preds):
            results.append((split, row, pred))
            logged.append({"image": path.as_posix(), "foreground": pred["foreground"], "background": pred["background"], "relation": pred["relation"], "latency": round(sec, 3)})
            tqdm.write(f"{split} {row['file']} {pred} {sec:.2f}s")
preds_df = pd.DataFrame(logged)
print(f"{preds_df['latency'].mean():.2f}s/image")
save_preds(model, preds_df)

ecommerce:   0%|          | 0/19 [00:00<?, ?it/s]

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
ecommerce:   5%|▌         | 1/19 [00:07<02:09,  7.19s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


ecommerce A_black_coffee_mug_on_an_office_desk_with_a_blurred_laptop_in_the_background.jpg {'relation': 'object_with_background', 'foreground': 'coffee_mug', 'background': 'blurred_laptop'} 3.59s
ecommerce a_camera_is_attached_to_a_tripod_with_a_mountain_in_the_background.jpg {'relation': 'object_with_background', 'foreground': 'camera', 'background': 'mountain'} 3.59s


ecommerce:  11%|█         | 2/19 [00:12<01:47,  6.32s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


ecommerce a_camera_on_a_tripod_with_a_mountain_in_the_background.jpg {'relation': 'object_with_background', 'foreground': 'camera', 'background': 'mountain'} 2.85s
ecommerce A_close_up_shot_of_a_fountain_pen_with_an_ink_bottle_highlighted_by_warm_bokeh_lights_in_the_background.jpg {'relation': 'object_with_background', 'foreground': 'pen', 'background': 'blurred_lights'} 2.85s


...

news:  88%|████████▊ | 7/8 [00:34<00:04,  4.71s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


news Joe_Biden_at_State_of_the_Union.jpg {'relation': 'people_at_location', 'foreground': 'Joe_Biden', 'background': 'White_House'} 2.06s
news Joe_Biden_at_White_House_Correspondents_Dinner.jpg {'relation': 'people_at_location', 'foreground': 'Joe_Biden', 'background': 'White_House'} 2.06s


news: 100%|██████████| 8/8 [00:38<00:00,  4.80s/it]


news Mr_Bezos_at_Florida.jpg {'foreground': '', 'background': '', 'relation': ''} 4.33s
2.52s/image


/tmp/ipykernel_3182/386134857.py:66: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  old = pd.concat([old[old.model_name != model], pd.DataFrame([row])], ignore_index=True)


In [13]:
from sklearn.metrics import classification_report

def pick(value, choices):
    text = str(value).strip()
    if text in choices:
        return text
    for part in text.replace(",", " ").split():
        if part in choices:
            return part
    return ""

keys = ("foreground", "background", "relation")
for split in ("ecommerce", "news"):
    pairs = [(pred, row) for name, row, pred in results if name == split]
    labels = {key: sorted({row[key] for _, row in pairs}) for key in keys}
    pairs = [({key: pick(pred.get(key, ""), labels[key]) for key in keys}, row) for pred, row in pairs]
    exact = sum(all(pred[k] == row[k] for k in keys) for pred, row in pairs) / len(pairs)
    accuracy = sum(pred[k] == row[k] for pred, row in pairs for k in keys) / (len(pairs) * len(keys))
    print(split, "exact", round(exact, 3), "accuracy", round(accuracy, 3))
    for key in keys:
        gold = [row[key] for _, row in pairs]
        pred = [item[key] for item, _ in pairs]
        print(key)
        print(classification_report(gold, pred, labels=labels[key], zero_division=0))

ecommerce exact 0.763 accuracy 0.921
foreground
                precision    recall  f1-score   support

   alarm_clock       1.00      1.00      1.00         1
      backpack       1.00      1.00      1.00         1
          book       1.00      1.00      1.00         1
        camera       1.00      1.00      1.00         9
    coffee_mug       0.89      1.00      0.94         8
           cup       0.00      0.00      0.00         1
   flower_vase       1.00      1.00      1.00         1
       handbag       1.00      1.00      1.00         1
           jar       1.00      1.00      1.00         1
        kettle       1.00      1.00      1.00         1
          lamp       1.00      1.00      1.00         1
        laptop       1.00      1.00      1.00         1
           pen       1.00      1.00      1.00         1
perfume_bottle       1.00      1.00      1.00         1
         phone       1.00      1.00      1.00         2
          soap       1.00      1.00      1.00         1

## Colab T4: Qwen3-VL-8B, 4-bit, structured output

In [6]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

COLAB_MODEL = "Qwen/Qwen3-VL-8B-Instruct"
colab_model = AutoModelForImageTextToText.from_pretrained(
    COLAB_MODEL,
    cache_dir="models",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
    device_map="auto",
)
colab_processor = AutoProcessor.from_pretrained(COLAB_MODEL, cache_dir="models", max_pixels=401408)
colab_processor.tokenizer.padding_side = "left"

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [7]:
def predict_colab(paths, vocab):
    Output = detection_model(vocab)
    tools = [{
        "type": "function",
        "function": {
            "name": "label_image",
            "description": "Choose one foreground, one background, and one relation.",
            "parameters": Output.model_json_schema(),
        },
    }]
    messages = [[{
        "role": "user",
        "content": [
            {"type": "image", "image": str(path.resolve())},
            {"type": "text", "text": (
                "Label this image. When two labels both fit, choose the more specific one.\n"
                f"foreground: {', '.join(vocab['foreground'])}\n"
                f"background: {', '.join(vocab['background'])}\n"
                f"relation: {', '.join(vocab['relation'])}"
            )},
        ],
    }] for path in paths]
    inputs = colab_processor.apply_chat_template(
        messages, tools=tools, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt", padding=True,
    ).to(colab_model.device)
    with torch.inference_mode():
        generated = colab_model.generate(
            **inputs, max_new_tokens=180, do_sample=False,
            stop_strings=["</tool_call>"], tokenizer=colab_processor.tokenizer,
        )
    texts = colab_processor.batch_decode(
        [out[len(src):] for src, out in zip(inputs.input_ids, generated)],
        skip_special_tokens=True,
    )
    preds = []
    for text in texts:
        blob = text[text.find("{"):text.rfind("}") + 1]
        try:
            obj = json.loads(blob).get("arguments", {})
            if isinstance(obj, str):
                obj = json.loads(obj)
            preds.append(Output.model_validate(obj).model_dump(mode="json"))
        except Exception:
            preds.append({key: "" for key in ("foreground", "background", "relation")})
    return preds

In [8]:
batch_size = 4
model = "qwen3-vl-8b_4bit_colab"
load_preds()
results, logged = [], []
for split in ("ecommerce", "news"):
    rows = json.loads(Path(f"data/{split}/gt_images.json").read_text())
    vocab = {k: sorted({r[k] for r in rows}) for k in ("foreground", "background", "relation")}
    groups = [rows[i:i + batch_size] for i in range(0, len(rows), batch_size)]
    for group in tqdm(groups, desc=split):
        paths = [Path(f"data/{split}/images") / row["file"] for row in group]
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        preds = predict_colab(paths, vocab)
        torch.cuda.synchronize()
        sec = (time.perf_counter() - t0) / len(group)
        for path, row, pred in zip(paths, group, preds):
            results.append((split, row, pred))
            logged.append({"image": path.as_posix(), "foreground": pred["foreground"], "background": pred["background"], "relation": pred["relation"], "latency": round(sec, 3)})
            tqdm.write(f"{split} {row['file']} {pred} {sec:.2f}s")
preds_df = pd.DataFrame(logged)
print(f"{preds_df['latency'].mean():.2f}s/image")
save_preds(model, preds_df)

ecommerce:   0%|          | 0/10 [00:00<?, ?it/s][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(
ecommerce:  10%|█         | 1/10 [01:17<11:34, 77.21s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


ecommerce A_black_coffee_mug_on_an_office_desk_with_a_blurred_laptop_in_the_background.jpg {'relation': 'object_with_background', 'foreground': 'coffee_mug', 'background': 'blurred_laptop'} 19.30s
ecommerce a_camera_is_attached_to_a_tripod_with_a_mountain_in_the_background.jpg {'relation': 'object_with_background', 'foreground': 'camera', 'background': 'mountain'} 19.30s
ecommerce a_camera_on_a_tripod_with_a_mountain_in_the_background.jpg {'relation': 'object_with_background', 'foreground': 'camera', 'background': 'mountain'} 19.30s
ecommerce A_close_up_shot_of_a_fountain_pen_with_an_ink_bottle_highlighted_by_warm_bokeh_lights_in_the_background.jpg {'relation': 'object_with_background', 'foreground': 'pen', 'background': 'blurred_lights'} 19.30s


ecommerce:  20%|██        | 2/10 [02:20<09:13, 69.18s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


ecommerce A_hand_holds_a_mushroom_themed_cup_indoors_focusing_on_coziness_with_a_blurred_screen_in_the_background.jpg {'relation': 'object_with_background', 'foreground': 'coffee_mug', 'background': 'blurred_laptop'} 15.89s
ecommerce A_hiking_backpack_resting_on_rocky_terrain_with_Tatra_Mountains_in_the_background.jpg {'relation': 'object_with_background', 'foreground': 'backpack', 'background': 'mountain'} 15.89s
ecommerce A_kettle_on_the_stove_with_a_teenage_boy_reading_a_book_in_the_background.jpg {'relation': 'object_with_background', 'foreground': 'kettle', 'background': 'woman'} 15.89s
ecommerce A_Mobile_phone_charging_on_a_marble_table_with_a_pen_wallet_and_note_book_in_the_background.jpg {'relation': 'object_with_background', 'foreground': 'phone', 'background': 'dessert'} 15.89s


ecommerce:  30%|███       | 3/10 [03:37<08:26, 72.42s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


...

news Brad_Pitt_at_Venice_Film_Festival.jpg {'relation': 'people_at_event', 'foreground': 'Brad_Pitt', 'background': 'Venice_Film_Festival'} 16.27s
news Donald_Trump_at_NRCC.jpg {'relation': 'people_at_event', 'foreground': 'Donald_Trump', 'background': 'NRCC'} 16.27s
news Donald_Trump_at_Oval_Office2.jpg {'relation': 'people_at_event', 'foreground': 'Donald_Trump', 'background': 'White_House'} 16.27s
news Donald_Trump_at_Oval_Office3.jpg {'relation': 'people_at_location', 'foreground': 'Donald_Trump', 'background': 'White_House'} 16.27s


news:  75%|███████▌  | 3/4 [02:37<00:54, 54.05s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


news Donald_Trump_at_Panama_City.jpg {'relation': 'people_at_event', 'foreground': 'Donald_Trump', 'background': 'White_House'} 13.52s
news Donald_Trump_at_White_House9.jpg {'relation': 'people_at_event', 'foreground': 'Donald_Trump', 'background': 'White_House'} 13.52s
news Joe_Biden_at_Buffalo.jpg {'relation': 'people_at_event', 'foreground': 'Joe_Biden', 'background': 'White_House'} 13.52s
news Joe_Biden_at_Oval_Office.jpg {'relation': 'people_at_location', 'foreground': 'Joe_Biden', 'background': 'White_House'} 13.52s


news: 100%|██████████| 4/4 [03:21<00:00, 50.36s/it]


news Joe_Biden_at_State_of_the_Union.jpg {'relation': 'people_at_event', 'foreground': 'Joe_Biden', 'background': 'State_of_the_Union'} 14.68s
news Joe_Biden_at_White_House_Correspondents_Dinner.jpg {'relation': 'people_at_event', 'foreground': 'Barack_Obama', 'background': 'White_House_Correspondents_Dinner'} 14.68s
news Mr_Bezos_at_Florida.jpg {'foreground': '', 'background': '', 'relation': ''} 14.68s
15.87s/image


In [9]:
from sklearn.metrics import classification_report

def pick(value, choices):
    text = str(value).strip()
    if text in choices:
        return text
    for part in text.replace(",", " ").split():
        if part in choices:
            return part
    return ""

keys = ("foreground", "background", "relation")
for split in ("ecommerce", "news"):
    pairs = [(pred, row) for name, row, pred in results if name == split]
    labels = {key: sorted({row[key] for _, row in pairs}) for key in keys}
    pairs = [({key: pick(pred.get(key, ""), labels[key]) for key in keys}, row) for pred, row in pairs]
    exact = sum(all(pred[k] == row[k] for k in keys) for pred, row in pairs) / len(pairs)
    accuracy = sum(pred[k] == row[k] for pred, row in pairs for k in keys) / (len(pairs) * len(keys))
    print(split, "exact", round(exact, 3), "accuracy", round(accuracy, 3))
    for key in keys:
        gold = [row[key] for _, row in pairs]
        pred = [item[key] for item, _ in pairs]
        print(key)
        print(classification_report(gold, pred, labels=labels[key], zero_division=0))

ecommerce exact 0.789 accuracy 0.912
foreground
                precision    recall  f1-score   support

   alarm_clock       1.00      1.00      1.00         1
      backpack       1.00      1.00      1.00         1
          book       1.00      1.00      1.00         1
        camera       1.00      1.00      1.00         9
    coffee_mug       0.89      1.00      0.94         8
           cup       0.00      0.00      0.00         1
   flower_vase       1.00      1.00      1.00         1
       handbag       1.00      1.00      1.00         1
           jar       1.00      1.00      1.00         1
        kettle       1.00      1.00      1.00         1
          lamp       1.00      1.00      1.00         1
        laptop       1.00      1.00      1.00         1
           pen       1.00      1.00      1.00         1
perfume_bottle       1.00      1.00      1.00         1
         phone       1.00      1.00      1.00         2
          soap       1.00      1.00      1.00         1